# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MirAziz0/flyrank1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one content item (`content_hash_id`) within one client (`client_hash_id`), with its daily GSC/GA4 metrics aggregated across a chosen window. The underlying warehouse table's own grain is finer — `report_date x client_hash_id x content_hash_id` — and I aggregate up to content-item grain for feature building (verified with a grain probe below).

**Time window:** I develop on the mid-panel month partition **`month=2026-03`** (all of March 2026), per the lane guide's warning to never develop label logic on the `_sample` table (that table is the panel's final month, June 2026 — the natural outcome window of any past-to-future label, so iterating there means peeking at the future). Inside March I use a mid-month **decision point of 2026-03-16**, splitting the month into:

- a **feature window**: 2026-03-01 to 2026-03-15 (15 days) — everything I'm allowed to know at decision time;
- a **label window**: 2026-03-16 to 2026-03-31 (16 days) — the future outcome the label is built from.

This keeps the whole exercise inside one partition (cheap to scan, no cross-month join needed) while still respecting a genuine past-to-future split. The full-scale production version of this label (prior 90 days -> next 30 days) needs multiple month partitions and is future work; June 2026 stays a sealed test month I do not touch here.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import getpass
import duckdb
import pandas as pd

# Token order: env var (local .env / shell) -> Colab Secret -> prompt (last resort).
# NEVER paste the token itself into a cell — this repo is public.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

DECISION_POINT = "2026-03-16"  # feature window ends 03-15, label window is 03-16-03-31
print("Connected. Querying month=2026-03 partition only (mid-panel month, per the lane guide).")


Connected. Querying month=2026-03 partition only (mid-panel month, per the lane guide).


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (knowable at the 2026-03-16 decision point, first-half window only):**

- `impressions_h1` — `SUM(gsc_impressions)` for `report_date <= 2026-03-15` where `gsc_data_available IS TRUE`
- `clicks_h1` — `SUM(gsc_clicks)`, same window and filter
- `avg_position_h1` — `SUM(gsc_sum_position) / SUM(gsc_impressions)`, same window (a volume-weighted average, not a naive mean of daily averages)
- `sessions_h1` — `SUM(ga4_sessions)` where `ga4_data_available IS TRUE`, same window
- `content_age_days_at_decision` — `2026-03-16 minus dim_content.content_created_date`

**Label / proxy:**

- `is_declining_h2` — did the page's average *daily* impressions in the second half (03-16 to 03-31) fall below 80% of its average daily impressions in the first half? Built from `impressions_h2` (second-half `SUM(gsc_impressions)`), which itself is never allowed to become a feature.

**Context (join/group/split only, never model inputs):**

- `client_hash_id`, `content_hash_id` — pseudonymized keys, used only to aggregate rows and to join `dim_content`.

**Excluded, with why:**

- **GA4/session columns on rows where `ga4_data_available IS FALSE`** — these are zero-filled placeholders for "tracking hadn't started yet," not real zero engagement. Using them as measured zeros would quietly penalize newer/late-onboarded clients.
- **GSC columns on rows where `gsc_data_available IS FALSE`** — same reasoning on the search side.
- **`impressions_h2` (and any other second-half/label-window column) as a feature** — this is the exact thing the label is computed from. Using it as an input would be a circular result: the model would just be re-reading its own answer, not discovering anything. This is deliberately demonstrated as "the trap" below, then removed.
- **`dim_content.keyword_hash_id` / `url_hash_id`** — scrambled raw-origin identifiers; useful only for grouping/dedup if ever needed, never as a feature (the originals are gone, and treating a random-looking hash as informative would be nonsense).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

FIELD_CONTRACT = {
    "feature": ["impressions_h1", "clicks_h1", "avg_position_h1", "sessions_h1", "content_age_days_at_decision"],
    "label_proxy": ["is_declining_h2 (derived from impressions_h2)"],
    "context": ["client_hash_id", "content_hash_id"],
    "excluded": [
        "ga4_*/session columns where ga4_data_available IS FALSE",
        "gsc_* columns where gsc_data_available IS FALSE",
        "impressions_h2 and any other label-window column, as a feature",
        "dim_content.keyword_hash_id / url_hash_id, as a feature",
    ],
}
for bucket, fields in FIELD_CONTRACT.items():
    print(f"{bucket}: {fields}")


feature: ['impressions_h1', 'clicks_h1', 'avg_position_h1', 'sessions_h1', 'content_age_days_at_decision']
label_proxy: ['is_declining_h2 (derived from impressions_h2)']
context: ['client_hash_id', 'content_hash_id']
excluded: ['ga4_*/session columns where ga4_data_available IS FALSE', 'gsc_* columns where gsc_data_available IS FALSE', 'impressions_h2 and any other label-window column, as a feature', 'dim_content.keyword_hash_id / url_hash_id, as a feature']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Three verification queries on `month=2026-03`, then a five-feature frame, then the leakage trap — all on the same mid-panel month.

### 3a. Grain: does one row really mean one (report_date, client, content)?

If the grain claim holds, grouping by those three columns and asking for groups with more than one row should return **nothing**.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

grain_probe = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM {MONTH}
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()

print("Duplicate (report_date, client_hash_id, content_hash_id) groups found:", len(grain_probe))
grain_probe


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (report_date, client_hash_id, content_hash_id) groups found: 0


,report_date,client_hash_id,content_hash_id,c


### 3b. Row count and date span of my slice

The contract claims a full calendar month, March 2026.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

span = con.sql(f"""
    SELECT COUNT(*) AS n_rows, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {MONTH}
""").df()
span


,n_rows,min_date,max_date
0,9841378,2026-03-01,2026-03-31


### 3c. Availability: filter with `IS TRUE`, show how many rows survive

`gsc_data_available` and `ga4_data_available` are the flags that separate "measured zero" from "no tracking yet." Filtering with `IS TRUE` (not just truthy) shows exactly how much of the month each source actually covers.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows
    FROM {MONTH}
""").df()

availability["gsc_available_pct"] = (availability["gsc_available_rows"] / availability["total_rows"]).round(3)
availability["ga4_available_pct"] = (availability["ga4_available_rows"] / availability["total_rows"]).round(3)
availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,gsc_available_rows,ga4_available_rows,gsc_available_pct,ga4_available_pct
0,9841378,3611061.0,413966.0,0.367,0.042


### 3d. Five features, built from the first-half (feature) window only

One line per feature: knowable at the 2026-03-16 decision moment because...

1. **`impressions_h1`** — knowable because it only sums `gsc_impressions` from days already logged (03-01 to 03-15), filtered to rows the pipeline has actually marked `gsc_data_available IS TRUE`.
2. **`clicks_h1`** — same reasoning: only past, already-synced GSC days.
3. **`avg_position_h1`** — a volume-weighted average position (`SUM(gsc_sum_position) / SUM(gsc_impressions)`) over the same past window; position on 03-16+ is not touched.
4. **`sessions_h1`** — knowable because it only sums `ga4_sessions` from the same past window, filtered to `ga4_data_available IS TRUE` (so clients without GA4 tracking yet aren't silently scored as zero engagement).
5. **`content_age_days_at_decision`** — knowable because `content_created_date` is a fact about the past (when the page was published), always available before any decision date that comes after it.

I also pull `impressions_h2` (second-half impressions) here **only** so I can build the label and demonstrate the leakage trap next — it is explicitly excluded from the feature list above.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

feat = con.sql(f"""
    WITH agg AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available THEN gsc_impressions ELSE 0 END) AS impressions_h1,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available THEN gsc_clicks ELSE 0 END) AS clicks_h1,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' AND gsc_data_available THEN gsc_sum_position ELSE 0 END) AS sum_position_h1,
            SUM(CASE WHEN report_date <= DATE '2026-03-15' AND ga4_data_available THEN ga4_sessions ELSE 0 END) AS sessions_h1,
            SUM(CASE WHEN report_date > DATE '2026-03-15' AND gsc_data_available THEN gsc_impressions ELSE 0 END) AS impressions_h2
        FROM {MONTH}
        GROUP BY 1, 2
    )
    SELECT * FROM agg WHERE impressions_h1 >= 50   -- minimum volume, keeps the label from being pure noise
""").df()

feat["avg_position_h1"] = feat["sum_position_h1"] / feat["impressions_h1"]

content = con.sql(f"SELECT content_hash_id, content_created_date FROM {CONTENT}").df()
feat = feat.merge(content, on="content_hash_id", how="left")
feat["content_age_days_at_decision"] = (
    pd.Timestamp(DECISION_POINT) - pd.to_datetime(feat["content_created_date"])
).dt.days

print("Feature frame shape:", feat.shape)
feature_cols = ["impressions_h1", "clicks_h1", "avg_position_h1", "sessions_h1", "content_age_days_at_decision"]
feat[["client_hash_id", "content_hash_id"] + feature_cols].head(8)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame shape: (92548, 10)


,client_hash_id,content_hash_id,impressions_h1,clicks_h1,avg_position_h1,sessions_h1,content_age_days_at_decision
0,client_73cda7b4e4f265ea,content_1fc8eecfb89547ec,660.0,0.0,5.866667,0.0,151
1,client_73cda7b4e4f265ea,content_2370b2f1a84409cc,559.0,2.0,5.475850,0.0,151
2,client_73cda7b4e4f265ea,content_da7ea4af9259c531,1347.0,10.0,2.976986,0.0,151
3,client_73cda7b4e4f265ea,content_950d07f1aaa3060c,1195.0,2.0,3.766527,0.0,151
4,client_73cda7b4e4f265ea,content_d438d66dee73b784,649.0,2.0,3.517720,0.0,151
5,client_73cda7b4e4f265ea,content_92ddb775e2a3a67b,710.0,0.0,3.604225,0.0,151
6,client_73cda7b4e4f265ea,content_a128f40fa2151fab,1556.0,2.0,6.000000,0.0,151
7,client_73cda7b4e4f265ea,content_ba6eb230c2842fa5,2169.0,11.0,3.561088,0.0,151


### 3e. The trap: add one label-derived column on purpose

Label: `is_declining_h2` = did average *daily* impressions drop by more than 20% from the first half to the second half of March? First I train a quick, honest logistic regression on the five real features. Then — on purpose — I add `impressions_h2` (part of the exact formula the label is built from) as a sixth "feature," retrain, and watch the score jump toward perfect. Then I delete it and keep the honest number.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

feat["avg_daily_h1"] = feat["impressions_h1"] / 15.0
feat["avg_daily_h2"] = feat["impressions_h2"] / 16.0
feat["is_declining_h2"] = (feat["avg_daily_h2"] < 0.8 * feat["avg_daily_h1"]).astype(int)
model_data = feat.dropna(subset=feature_cols + ["is_declining_h2"])

print("Positive rate (is_declining_h2):", round(model_data["is_declining_h2"].mean(), 3))

X, y = model_data[feature_cols], model_data["is_declining_h2"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_tr)
honest_model = LogisticRegression(max_iter=1000).fit(scaler.transform(X_tr), y_tr)
honest_auc = roc_auc_score(y_te, honest_model.predict_proba(scaler.transform(X_te))[:, 1])
print(f"HONEST ROC AUC (5 real features only): {honest_auc:.3f}")


Positive rate (is_declining_h2): 0.327
HONEST ROC AUC (5 real features only): 0.603


In [8]:
# SPRINGING THE TRAP ON PURPOSE: adding impressions_h2 — part of the label's own formula — as a "feature."
# Do not do this in a real pipeline. This cell exists only to show what leakage looks like.

leaky_cols = feature_cols + ["impressions_h2"]
Xl, yl = model_data[leaky_cols], model_data["is_declining_h2"]
Xl_tr, Xl_te, yl_tr, yl_te = train_test_split(Xl, yl, test_size=0.25, random_state=42, stratify=yl)
scaler_leak = StandardScaler().fit(Xl_tr)
leaky_model = LogisticRegression(max_iter=1000).fit(scaler_leak.transform(Xl_tr), yl_tr)
leaky_auc = roc_auc_score(yl_te, leaky_model.predict_proba(scaler_leak.transform(Xl_te))[:, 1])

print(f"LEAKY ROC AUC (+ impressions_h2, the label's own ingredient): {leaky_auc:.3f}")
print(f"Jump: {honest_auc:.3f} -> {leaky_auc:.3f} just from adding one label-derived column.")


LEAKY ROC AUC (+ impressions_h2, the label's own ingredient): 0.998
Jump: 0.603 -> 0.998 just from adding one label-derived column.


**Deleting the leak, keeping the honest number.** `impressions_h2` is removed from the feature set below (it was only ever added to demonstrate the trap). The number I actually report and carry forward is the honest one from 3e's first model, not the leaky one — the leaky AUC is evidence of a mistake, not a result to be proud of.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

assert "impressions_h2" not in feature_cols  # the leak never lived in the real feature list
del leaky_cols, Xl, yl, Xl_tr, Xl_te, yl_tr, yl_te, scaler_leak, leaky_model  # drop the leaky objects

print("Final, honest result carried forward:")
print(f"  Precision benchmark for this week's exercise -> ROC AUC = {honest_auc:.3f} (5 real features, no leakage)")


Final, honest result carried forward:
  Precision benchmark for this week's exercise -> ROC AUC = 0.603 (5 real features, no leakage)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation: this is an unbalanced panel, and my one-month slice cannot show that on its own.** `dim_clients.gsc_data_start` / `ga4_data_start` differ per client — some clients have been tracked since the panel began (2025-01-27), others only joined recently — so a client with a short history contributes fewer, more recent rows to any given month than a long-tenured client. The availability query above (3c) already hints at this at the row level: only 36.7% of March rows have `gsc_data_available IS TRUE` and just 4.2% have `ga4_data_available IS TRUE`, which is consistent with a large share of client-days simply predating that client's tracking start — not real "zero traffic." A single month's numbers cannot separate "this page genuinely has no search demand" from "this client's GSC/GA4 sync had not started yet," and I have not (yet) joined `dim_clients` to confirm the split client-by-client — that is the natural next check before trusting this slice for anything beyond this week's exercise.

Two smaller, related limits worth naming honestly:

- **The label here is a short-window proxy, not the production label.** A 15-day-vs-16-day within-month split is a stand-in built to fit inside one partition; the real capstone target (prior 90 days -> next 30 days) needs multiple month partitions and has not been built yet.
- **One month is not validated across months.** Feature-label relationships found in March may not hold in April or in the sealed June test month — that comparison is future work, not something this notebook claims.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
clients = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM {CLIENTS}
""").df()

clients["gsc_data_start"] = pd.to_datetime(clients["gsc_data_start"])
clients["ga4_data_start"] = pd.to_datetime(clients["ga4_data_start"])

march_start = pd.Timestamp("2026-03-01")
print("Total clients in dim_clients:", len(clients))
print("Clients whose GSC tracking started AFTER 2026-03-01 (no full March history):",
      (clients["gsc_data_start"] > march_start).sum())
print("Clients whose GA4 tracking started AFTER 2026-03-01 (no full March history):",
      (clients["ga4_data_start"] > march_start).sum())
print("Clients with NO ga4_data_start at all (never had GA4 access):",
      clients["ga4_data_start"].isna().sum())


Total clients in dim_clients: 104
Clients whose GSC tracking started AFTER 2026-03-01 (no full March history): 15
Clients whose GA4 tracking started AFTER 2026-03-01 (no full March history): 25
Clients with NO ga4_data_start at all (never had GA4 access): 53


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.